# Dataset Characteristics Analysis

This notebook analyzes the characteristics of the evaluation dataset, which consists of projects from three sources:

1. **EqBench**: Programs from the EqBench benchmark designed for automated reasoning tools, with EvoSuite-generated test suites at different timeout settings (1s, 10s, 60s)
2. **Apache Commons Utils**: Utility methods extracted from Apache Commons projects, including both developer-written tests and EvoSuite-generated test suites
3. **RepoReapers**: Open source Java projects from the RepoReapers dataset, selected based on size, structure, and build tool criteria

The analysis provides file counts, class counts, source lines of code (SLOC), and test method statistics for each project.

In [1]:
# Import dataset analysis functions
from teralizer.dataset_characteristics import (
    get_dataset_statistics,
    generate_dataset_table,
    generate_dataset_csv_data,
)
from teralizer.config import db_config
from teralizer.exclusions import get_excluded_project_names
from teralizer.exports import save_latex_table, save_csv_data
from IPython.display import display

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Data Collection

Collect statistics for all projects in the evaluation dataset.

In [2]:
# Get database connections
conn_dev = db_config.get_dev_engine()
conn_test = db_config.get_test_engine()

# Get excluded project names from both databases
excluded_projects_dev = get_excluded_project_names(conn_dev)
excluded_projects_test = get_excluded_project_names(conn_test)
excluded_projects = excluded_projects_dev.union(excluded_projects_test)
print(f"Excluding {len(excluded_projects)} projects from dataset statistics")

# Get dataset statistics (computes fresh if projects/ available, else loads pre-computed)
project_aggregates = get_dataset_statistics(conn_dev, conn_test, excluded_projects)

# Display aggregated statistics
display(project_aggregates)

Excluding 529 projects from dataset statistics
Projects directory not found. Loading pre-computed reference statistics...
Note: For full replication, download teralizer-projects-*.zip from Zenodo


,project,main_files,main_classes,main_sloc,test_files,test_classes,test_sloc,test_methods
0,commons-utils,106,247,19709,80,119,14389,725
1,commons-utils-es-default-10s,106,247,19709,103,103,19082,2738
2,commons-utils-es-default-1s,106,247,19709,103,103,17524,2481
3,commons-utils-es-default-60s,106,247,19709,102,102,18839,2735
4,eqbench-es-default-60s,544,652,27871,544,544,37836,4974
5,eqbench-es-default-10s,544,652,27871,543,543,36937,4875
6,eqbench-es-default-1s,544,652,27871,544,544,35666,4718
7,repo-reapers (total),41292,50474,2735127,22281,30894,2012601,81810
8,repo-reapers (mean),65,79,4320,35,48,3179,162
9,repo-reapers (median),49,56,3253,23,26,2107,86


## LaTeX Table Generation

Generate the dataset characteristics table for paper inclusion.

In [3]:
# Generate LaTeX table
latex_table = generate_dataset_table(project_aggregates)
print(latex_table)

# Save LaTeX table
save_latex_table(latex_table, "tab-dataset-statistics")

\begin{table}[H]
  \caption{Number of files, classes, source lines of code (SLOC), and test methods per project.}
  \label{tab:dataset-statistics}
  \begin{tabular}{lrrrrrrr}
    \toprule
    & \multicolumn{3}{r}{Implementation} & \multicolumn{4}{r}{Test} \\
    \cmidrule(lr){2-4} \cmidrule(lr){5-8}
    Project & Files & Classes & SLOC & Files & Classes & SLOC & Methods \\
    \midrule
    \DatasetEqBenchA{} & 544 & 652 & 27,871 & 544 & 544 & 35,666 & 4,718 \\
    \DatasetEqBenchB{} & 544 & 652 & 27,871 & 543 & 543 & 36,937 & 4,875 \\
    \DatasetEqBenchC{} & 544 & 652 & 27,871 & 544 & 544 & 37,836 & 4,974 \\
    \midrule
    \DatasetCommonsA{} & 106 & 247 & 19,709 & 103 & 103 & 17,524 & 2,481 \\
    \DatasetCommonsB{} & 106 & 247 & 19,709 & 103 & 103 & 19,082 & 2,738 \\
    \DatasetCommonsC{} & 106 & 247 & 19,709 & 102 & 102 & 18,839 & 2,735 \\
    \midrule
    \DatasetCommonsDev{} & 106 & 247 & 19,709 & 80 & 119 & 14,389 & 725 \\
    \midrule
    \DatasetRepoReapers{} (total) & 41,29

## CSV Data Export

Export dataset statistics as CSV for further analysis.

In [4]:
# Generate CSV data
csv_data = generate_dataset_csv_data(project_aggregates)

# Save CSV data
csv_path = save_csv_data(
    csv_data,
    "dataset-statistics-data",
    "Dataset statistics showing files, classes, SLOC, and test methods per project",
)

print(f"Dataset statistics exported to: {csv_path}")
print(f"Shape: {csv_data.shape}")
print("Sample data:")
display(csv_data.head())

Dataset statistics exported to: /app/analysis/output/verify/data/dataset-statistics-data.csv
Shape: (10, 8)
Sample data:


,project_name,implementation_files,implementation_classes,implementation_sloc,test_files,test_classes,test_sloc,test_methods
0,commons-utils,106,247,19709,80,119,14389,725
1,commons-utils-es-default-10s,106,247,19709,103,103,19082,2738
2,commons-utils-es-default-1s,106,247,19709,103,103,17524,2481
3,commons-utils-es-default-60s,106,247,19709,102,102,18839,2735
4,eqbench-es-default-60s,544,652,27871,544,544,37836,4974


## Dataset Summary

The evaluation dataset includes:

- **EqBench projects**: Programs designed for automated reasoning tools, avoiding features like recursion and reflection that challenge symbolic execution
- **Apache Commons projects**: Utility methods extracted from Apache Commons libraries, representing larger open source codebases
- **RepoReapers projects**: Selection of Java projects meeting specific size and structure criteria for broader applicability assessment

Each project type uses different test suite generation approaches (EvoSuite with varying timeouts, developer-written tests) to evaluate generalization effectiveness across different testing scenarios.